### Neighbors Imbalance Approach

In [1]:
import sys
from pathlib import Path
_here = Path().resolve()
if (_here / "src").is_dir():
    sys.path.insert(0, str(_here / "src"))
else:
    sys.path.insert(0, str(_here))

In [ ]:
import sklearn
import scipy
import numpy as np
from stroke_data import get_stroke_data_for_cv, get_stroke_data

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import GridSearchCV

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from all_baselines import run_all_baselines

In [3]:
X_train, X_test, y_train, y_test = get_stroke_data_for_cv("data/knn-standardize-distance.csv")

### Nearest Neighbor Label Swapping Approach

Artificially swap the labels of the k nearest neighbors to positive labels. 

If already positive, keep unchanged.

If negative, swap to positive. 

In [4]:
# find all instances of stroke having patients 
stroke_idx = []

for idx in range(len(y_train)):
    if y_train[idx] == 1:
        stroke_idx.append(idx)

In [5]:
# number of neighbors to consider 
K = 5

In [6]:
# get k nearest neighbors to the stroke-having subjects 

# for each positive element in the dataset 
for current_idx in stroke_idx:

    # get current element 
    current = X_train[current_idx]

    # get all euclidean distances btwn all points and current 
    dists = []
    for elem in X_train:
        dists.append(np.linalg.norm(current - elem))

    # set the current point's distance to positive inf 
    dists[current_idx] = np.inf

    # for each k, find the closest element, push it to positive inf, then flip the labels 
    k_neighbors = []
    for i in range(K):
        idx = np.argmin(dists)
        dists[idx] = np.inf

        # flip labels of the found indices -- if already positive do nothing 
        if y_train[idx] == 0:
            y_train[idx] = 1

In [9]:
# get the result from running all baselines with the balanced dataset
result = run_all_baselines(X_train, X_test, y_train, y_test)

In [10]:
from pprint import pprint 
pprint(result)

{'knn': {'test': {'accuracy': 0.7759295499021527,
                  'f1': 0.15498154981549817,
                  'precision': 0.09502262443438914,
                  'recall': 0.42},
         'train': {'accuracy': 1.0,
                   'f1': 1.0,
                   'precision': 1.0,
                   'recall': 1.0}},
 'lr': {'test': {'accuracy': 0.8581213307240705,
                 'f1': 0.2926829268292683,
                 'precision': 0.1935483870967742,
                 'recall': 0.6},
        'train': {'accuracy': 0.8116438356164384,
                  'f1': 0.4914134742404227,
                  'precision': 0.6009693053311793,
                  'recall': 0.41564245810055866}},
 'mlp': {'test': {'accuracy': 0.7974559686888454,
                  'f1': 0.16194331983805668,
                  'precision': 0.10152284263959391,
                  'recall': 0.4},
         'train': {'accuracy': 0.9931506849315068,
                   'f1': 0.984251968503937,
                   'precision': 

## New hyperparameters for neighbors

Run GridSearchCV with 5 fold cross validation to find the best hyperparameters for new models with the balanced dataset

In [11]:
# Logistic Regression
param_lr = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100, 1000],
            "solver": ["liblinear", "newton-cg", "newton-cholesky", "sag", "saga"]
            }

lr = LogisticRegression(random_state=42)
grid_search_lr = GridSearchCV(estimator=lr, param_grid=param_lr, scoring="f1") # , verbose=5)
grid_search_lr.fit(X=X_train, y=y_train)

best_lr_model = grid_search_lr.best_estimator_
print("Best params = ", grid_search_lr.best_params_)

best_lr_model.fit(X=X_train, y=y_train)

lr_preds_train = best_lr_model.predict(X_train)
lr_preds = best_lr_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, lr_preds_train))
print("F1 = ", f1_score(y_train, lr_preds_train))
print("Precision = ", precision_score(y_train, lr_preds_train))
print("Recall = ", recall_score(y_train, lr_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, lr_preds))
print("F1 = ", f1_score(y_test, lr_preds))
print("Precision = ", precision_score(y_test, lr_preds))
print("Recall = ", recall_score(y_test, lr_preds))

Best params =  {'C': 1.0, 'solver': 'saga'}
-- Train --
Accuracy =  0.8258317025440313
F1 =  0.5424164524421594
Precision =  0.6384266263237519
Recall =  0.47150837988826816
-- Test --
Accuracy =  0.8551859099804305
F1 =  0.3148148148148148
Precision =  0.20481927710843373
Recall =  0.68


In [12]:
# SVM
param_svm = {"C": [0.001, 0.01, 0.1, 1.0, 10.0],
             "kernel": ["linear", "rbf"], 
             "gamma": [0.01, 0.1, 1, 10, 100, "auto", "scale"],
             }

svm = SVC()
grid_search_svm = GridSearchCV(estimator=svm, param_grid=param_svm, scoring="f1") # , verbose=5)
grid_search_svm.fit(X=X_train, y=y_train)

best_svm_model = grid_search_svm.best_estimator_
print("Best params = ", grid_search_svm.best_params_)

best_svm_model.fit(X=X_train, y=y_train)

svm_preds_train = best_svm_model.predict(X_train)
svm_preds = best_svm_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, svm_preds_train))
print("F1 = ", f1_score(y_train, svm_preds_train))
print("Precision = ", precision_score(y_train, svm_preds_train))
print("Recall = ", recall_score(y_train, svm_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, svm_preds))
print("F1 = ", f1_score(y_test, svm_preds))
print("Precision = ", precision_score(y_test, svm_preds))
print("Recall = ", recall_score(y_test, svm_preds))

Best params =  {'C': 10.0, 'gamma': 1, 'kernel': 'rbf'}
-- Train --
Accuracy =  0.988013698630137
F1 =  0.9723632261703328
Precision =  0.9817767653758542
Recall =  0.9631284916201117
-- Test --
Accuracy =  0.8131115459882583
F1 =  0.15859030837004406
Precision =  0.1016949152542373
Recall =  0.36


In [13]:
# Random Forests
param_rf = {"n_estimators": [5, 10, 15, 20],
            "max_features": ["sqrt", "log2", None],
            "max_depth": [5, 10, 15, None],
            "max_leaf_nodes": [5, 10, 15, None],
            "bootstrap": [True, False] 
            }

rf = RandomForestClassifier()
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_rf, scoring="f1") # , verbose=5)
grid_search_rf.fit(X=X_train, y=y_train)

best_rf_model = grid_search_rf.best_estimator_
print("Best params = ", grid_search_rf.best_params_)

best_rf_model.fit(X=X_train, y=y_train)

rf_preds_train = best_rf_model.predict(X_train)
rf_preds = best_rf_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, rf_preds_train))
print("F1 = ", f1_score(y_train, rf_preds_train))
print("Precision = ", precision_score(y_train, rf_preds_train))
print("Recall = ", recall_score(y_train, rf_preds_train))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, rf_preds))
print("F1 = ", f1_score(y_test, rf_preds))
print("Precision = ", precision_score(y_test, rf_preds))
print("Recall = ", recall_score(y_test, rf_preds))

Best params =  {'bootstrap': False, 'max_depth': 15, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'n_estimators': 20}
-- Train --
Accuracy =  0.9987769080234834
F1 =  0.9972020145495244
Precision =  0.9988789237668162
Recall =  0.9955307262569832
-- Test --
Accuracy =  0.8101761252446184
F1 =  0.17796610169491525
Precision =  0.11290322580645161
Recall =  0.42


In [14]:
# XGBoost 
param_xgb = {
            "max_depth": [6, 10, 15, 20],
            "subsample": [0.1, 0.5, 1], # subsampling helps prevent overfitting. High subsampling number=high overfitting change
            "lambda": [0.5],
            "gamma": [0.5, 1 ,2],
            "objective": ["binary:logistic"],
            "eta": [0.1, 0.3, 1],
            }

xgb = XGBClassifier()
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_xgb, scoring="f1")#, verbose=3)
grid_search_xgb.fit(X=X_train, y=y_train)

best_xgb_model = grid_search_xgb.best_estimator_
print("Best params = ", grid_search_xgb.best_params_)

best_xgb_model.fit(X=X_train, y=y_train)


train_xgb_preds = best_xgb_model.predict(X_train)
xgb_preds = best_xgb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_xgb_preds))
print("F1 = ", f1_score(y_train, train_xgb_preds))
print("Precision = ", precision_score(y_train, train_xgb_preds))
print("Recall = ", recall_score(y_train, train_xgb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, xgb_preds))
print("F1 = ", f1_score(y_test, xgb_preds))
print("Precision = ", precision_score(y_test, xgb_preds))
print("Recall = ", recall_score(y_test, xgb_preds))

Best params =  {'eta': 0.1, 'gamma': 1, 'lambda': 0.5, 'max_depth': 15, 'objective': 'binary:logistic', 'subsample': 0.5}
-- Train --
Accuracy =  0.9770058708414873
F1 =  0.9461009174311926
Precision =  0.9717314487632509
Recall =  0.9217877094972067
-- Test --
Accuracy =  0.8277886497064579
F1 =  0.23478260869565218
Precision =  0.15
Recall =  0.54


In [15]:
# Naive Bayes
param_nb = {
            "var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7]   # default is 1e-9
            }

nb = GaussianNB()
grid_search_nb = GridSearchCV(estimator=nb, param_grid=param_nb, scoring="f1")#, verbose=3)
grid_search_nb.fit(X=X_train, y=y_train)

best_nb_model = grid_search_nb.best_estimator_
print("Best params = ", grid_search_nb.best_params_)

best_nb_model.fit(X=X_train, y=y_train)

train_nb_preds = best_nb_model.predict(X_train)
nb_preds = best_nb_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_nb_preds))
print("F1 = ", f1_score(y_train, train_nb_preds))
print("Precision = ", precision_score(y_train, train_nb_preds))
print("Recall = ", recall_score(y_train, train_nb_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, nb_preds))
print("F1 = ", f1_score(y_test, nb_preds))
print("Precision = ", precision_score(y_test, nb_preds))
print("Recall = ", recall_score(y_test, nb_preds))

Best params =  {'var_smoothing': 1e-11}
-- Train --
Accuracy =  0.8087084148727984
F1 =  0.5626398210290827
Precision =  0.5632698768197089
Recall =  0.5620111731843576
-- Test --
Accuracy =  0.8023483365949119
F1 =  0.2462686567164179
Precision =  0.15137614678899083
Recall =  0.66


In [16]:
# KNN
param_knn = {
    "n_neighbors": [1, 2, 3, 5, 10]
}

knn = KNeighborsClassifier()
grid_search_knn = GridSearchCV(estimator=knn, param_grid=param_knn, scoring="f1") #, verbose=5)
grid_search_knn.fit(X=X_train, y=y_train)

best_knn_model = grid_search_knn.best_estimator_
print("Best params = ", grid_search_knn.best_params_)

best_knn_model.fit(X=X_train, y=y_train)

train_knn_preds = best_knn_model.predict(X_train)
knn_preds = best_knn_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_knn_preds))
print("F1 = ", f1_score(y_train, train_knn_preds))
print("Precision = ", precision_score(y_train, train_knn_preds))
print("Recall = ", recall_score(y_train, train_knn_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, knn_preds))
print("F1 = ", f1_score(y_test, knn_preds))
print("Precision = ", precision_score(y_test, knn_preds))
print("Recall = ", recall_score(y_test, knn_preds))

Best params =  {'n_neighbors': 3}
-- Train --
Accuracy =  0.9515655577299413
F1 =  0.8918032786885246
Precision =  0.8727272727272727
Recall =  0.911731843575419
-- Test --
Accuracy =  0.7857142857142857
F1 =  0.18587360594795538
Precision =  0.1141552511415525
Recall =  0.5


In [17]:
# Neural Network
param_mlp = {"hidden_layer_sizes": [(100, 100, 100), (100, 100, 100, 100), (100, 100, 100, 100, 100)],
             "solver": ["adam", "sgd"],
             "alpha": [0.0001, 0.001, 0.01, 1.0],
             "max_iter": [200, 500, 1000, 1500]
            }

mlp = MLPClassifier(random_state=42)
grid_search_mlp = GridSearchCV(estimator=mlp, param_grid=param_mlp, scoring="f1")
grid_search_mlp.fit(X=X_train, y=y_train)

best_mlp_model = grid_search_mlp.best_estimator_
print("Best params = ", grid_search_mlp.best_params_)

best_mlp_model.fit(X=X_train, y=y_train)

train_mlp_preds = best_mlp_model.predict(X_train)
mlp_preds = best_mlp_model.predict(X_test)

print("-- Train --")
print("Accuracy = ", accuracy_score(y_train, train_mlp_preds))
print("F1 = ", f1_score(y_train, train_mlp_preds))
print("Precision = ", precision_score(y_train, train_mlp_preds))
print("Recall = ", recall_score(y_train, train_mlp_preds))

print("-- Test --")
print("Accuracy = ", accuracy_score(y_test, mlp_preds))
print("F1 = ", f1_score(y_test, mlp_preds))
print("Precision = ", precision_score(y_test, mlp_preds))
print("Recall = ", recall_score(y_test, mlp_preds))

/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_

Best params =  {'alpha': 0.001, 'hidden_layer_sizes': (100, 100, 100, 100, 100), 'max_iter': 200, 'solver': 'adam'}
-- Train --
Accuracy =  0.9828767123287672
F1 =  0.9601820250284414
Precision =  0.9779837775202781
Recall =  0.9430167597765363
-- Test --
Accuracy =  0.7984344422700587
F1 =  0.14166666666666666
Precision =  0.08947368421052632
Recall =  0.34
